# PCA

## Imports


In [ ]:
import os
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from datetime import date,datetime

In [ ]:
d = date.today()
t = datetime.now().time()

print(f'''
DATE: {d} at {t}

pandas=={pd.__version__}
numpy=={np.__version__}
plotly=={plotly.__version__}
''')

## Set directories and variables

#### Common paths

In [ ]:
# Directories

# Main directory
main_dir = "/path/to/home"
MAIN_DIR = main_dir     ### alias

# Data directory
data_dir = f"{main_dir}/input"
DATA_DIR = data_dir     ### alias

# Raw data directory
raw_dir = f"{data_dir}/raw"
RAW_DIR = raw_dir     ### alias

# Meta data (covariate, population, ancestry labels, etc.)
meta_dir = f"{data_dir}/meta"
META_DIR = meta_dir     ### alias

# Projected PCA directory
pca_dir = f"{raw_dir}/PCA"
PCA_DIR = pca_dir     ### alias
os.makedirs(pca_dir, exist_ok=True)

#### Paths to software and tools

In [ ]:
# Hestia NGS Software
tools = "/path/to/tools"

# Plink1.9 and Plink2.0 path
plink = f"{tools}/plink_linux_x86_64_20250615/plink"
plink2 = f"{tools}/plink2_linux_avx2_20250609/plink2"

# GCTA
gcta = f"{tools}/gcta_1.93.1beta/gcta64"

# PCA Projected path
pca_path = f"{tools}/ProjectedPCAAndModelSelection"

#### Input, output, covariate files

In [ ]:
# Input file (GWASQC unrelated output)
inputPfile = f'{pca_dir}/CATPD_Relationship'

# Covariate file
covar_path = f'{meta_dir}/CATPD_covariate_for_qc.cov'
pop_path = f'{meta_dir}/CATPD_covariate_for_qc.pop'

# GenoTools ancestry predictions
gt_path = f'{meta_dir}/CATPD_qc_ancestry_umap_linearsvc_predicted_labels.txt'

# Pheno Name
pheno = f"STATUS"

# Output prefix
prefix = f'CATPD_Unrelated'

# Threads
threads = 8

In [ ]:
# Empty file for remove samples if there's no outlier list
remove = pd.DataFrame([["NA", "NA"]])
samplesToRemove = f"{meta_dir}/samplestoremove.txt"
remove.to_csv(samplesToRemove, sep="\t", index=False, header=False, na_rep='NA')

### Get PROJECTED PCA

In [ ]:
%%time
createProjectedPCA = ["python3", f"{pca_path}/covarProjectedPCA.py",
                    "-A", inputPfile,                    # Should be plink1.9 bfile 
                    "-t", covar_path,
                    "-r", samplesToRemove,               # Should be an empty file with NA\tNA in case nothing to remove, otherwise error
                    "--threads", str(threads),
                    "-n", prefix,
                    "-f", pca_dir,
                    "--selectModel", f"{pca_path}/selectModel.R",
                    "--plink1", plink,
                    "--plink2", plink2,
                    "--gcta", gcta,]
d = date.today()
t = datetime.now().time()
print(f"{t} {d}: RUNNING PCA PROJECTED")
subprocess.run(createProjectedPCA, check=True, cwd=f"{tools}/SAIGE_pixi/SAIGE")

## Plots

In [ ]:
pca = pd.read_csv(f'{pca_dir}/{prefix}.tsv', sep='\t', header=0,
                 usecols=['IID', 
                          'SEX', 
                          'DISEASE', 
                          'GCTA_PC1', 
                          'GCTA_PC2', 
                          'GCTA_PC3',
                          'GCTA_PC4',
                         ]
                 ).rename(columns={'IID': 'ID'})

# Merge with popultions
pop = pd.read_csv(pop_path, sep='\t', header=0)
pca = pd.merge(pca, pop, on='ID', how='left')

# Merge with ancestry predictions
gt = pd.read_csv(gt_path, sep='\t', 
                 usecols=['IID', 
                          'label']
                ).rename(columns={'IID':'ID'})
pca = pd.merge(pca, gt, on='ID', how='left')

pca.head()

#### Define colors

In [ ]:
color_map = {
    'KAZ': '#636EFA',
    'AZE': '#EF553B',
    'GEO': '#00CC96',
    'ARM': '#AB63FA',
    'TJK': '#FFA15A',
}

In [ ]:
labels = gt['label'].sort_values().unique()
labels

In [ ]:
colors = px.colors.qualitative.Safe
label_color_map = {k: colors[i] for i, k in enumerate(labels)}
label_color_map

### Plot by populations

In [ ]:
fig1 = px.scatter(pca, x='GCTA_PC1', y='GCTA_PC2', color='POP')
fig2 = px.scatter(pca, x='GCTA_PC2', y='GCTA_PC3', color='POP')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('PC1 vs PC2', 'PC2 vs PC3'),
    horizontal_spacing=0.12,
)

seen = set()
for trace in fig1.data:
    fig.add_trace(trace, row=1, col=1)
    seen.add(trace.name)
for trace in fig2.data:
    trace.showlegend = trace.name not in seen
    fig.add_trace(trace, row=1, col=2)

fig.update_xaxes(showline=True, linewidth=0.5, linecolor='black', mirror=True,
                 ticks='outside', tickwidth=0.5, tickcolor='black')
fig.update_yaxes(showline=True, linewidth=0.5, linecolor='black', mirror=True,
                 ticks='outside', tickwidth=0.5, tickcolor='black')

fig.update_xaxes(title_text="PC1", row=1, col=1)
fig.update_yaxes(title_text="PC2", row=1, col=1)
fig.update_xaxes(title_text="PC2", row=1, col=2)
fig.update_yaxes(title_text="PC3", row=1, col=2)

# Remove legend from all real traces
for trace in fig.data:
    trace.showlegend = False

# Add invisible legend-only traces with square markers
for trace in fig1.data:
    fig.add_trace(dict(
        type='scatter',
        x=[None], y=[None],
        mode='markers',
        name=trace.name,
        marker=dict(
            symbol='square',
            size=12,
            color=trace.marker.color,
            line=dict(color='black', width=1),
        ),
        showlegend=True,
    ))
fig.update_layout(
    height=500,
    width=1050,
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=50, r=50, t=50, b=50),
    font=dict(family='Arial'),
    yaxis=dict(scaleanchor='x', scaleratio=1),
    yaxis2=dict(scaleanchor='x2', scaleratio=1),
)

fig.show()

In [ ]:
countries = pca['POP'].unique()
n_rows = len(countries)

fig = make_subplots(
    rows=n_rows, cols=2,
    subplot_titles=[f"{c} PC1 vs PC2" for c in countries] + [f"{c} PC2 vs PC3" for c in countries]
)

In [ ]:
fig1 = px.scatter(pca, x='GCTA_PC1', y='GCTA_PC2', color='POP')
fig2 = px.scatter(pca, x='GCTA_PC2', y='GCTA_PC3', color='POP')
fig3 = px.scatter(pca, x='GCTA_PC3', y='GCTA_PC4', color='POP')

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('PC1 vs PC2', 'PC2 vs PC3', 'PC3 vs PC4'),
    horizontal_spacing=0.08,
)

seen = set()
for trace in fig1.data:
    fig.add_trace(trace, row=1, col=1)
    seen.add(trace.name)
for trace in fig2.data:
    trace.showlegend = False
    fig.add_trace(trace, row=1, col=2)
for trace in fig3.data:
    trace.showlegend = False
    fig.add_trace(trace, row=1, col=3)

fig.update_xaxes(showline=True, linewidth=0.5, linecolor='black', mirror=True,
                 ticks='outside', tickwidth=0.5, tickcolor='black')
fig.update_yaxes(showline=True, linewidth=0.5, linecolor='black', mirror=True,
                 ticks='outside', tickwidth=0.5, tickcolor='black')

fig.update_xaxes(title_text="PC1", row=1, col=1)
fig.update_yaxes(title_text="PC2", row=1, col=1)
fig.update_xaxes(title_text="PC2", row=1, col=2)
fig.update_yaxes(title_text="PC3", row=1, col=2)
fig.update_xaxes(title_text="PC3", row=1, col=3)
fig.update_yaxes(title_text="PC4", row=1, col=3)

# Remove legend from all real traces
for trace in fig.data:
    trace.showlegend = False

# Add invisible legend-only traces with square markers
for trace in fig1.data:
    fig.add_trace(dict(
        type='scatter',
        x=[None], y=[None],
        mode='markers',
        name=trace.name,
        marker=dict(
            symbol='square',
            size=12,
            color=trace.marker.color,
            line=dict(color='black', width=1),
        ),
        showlegend=True,
    ))

fig.update_layout(
    height=500,
    width=1500,
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=50, r=50, t=50, b=50),
    font=dict(family='Arial'),
    yaxis=dict(scaleanchor='x', scaleratio=1),
    yaxis2=dict(scaleanchor='x2', scaleratio=1),
    yaxis3=dict(scaleanchor='x3', scaleratio=1),
)

fig.show()

In [ ]:
countries = ['KAZ', 'TJK', 'ARM', 'AZE', 'GEO']
n = len(countries)


fig = make_subplots(
    rows=n, cols=3,
    subplot_titles=[f"{c}: PC{a} vs PC{b}" for c in countries for a, b in [(1,2),(2,3),(3,4)]],
    horizontal_spacing=0.1,
    vertical_spacing=0.08,
)

# Get consistent color mapping from full dataset
full_fig = px.scatter(pca, x='GCTA_PC1', y='GCTA_PC2', color='POP')
color_map = {trace.name: trace.marker.color for trace in full_fig.data}

for row_idx, country in enumerate(countries, start=1):
    subset = pca[pca['POP'] == country]
    color = color_map.get(country, 'grey')

    for col_idx, (xpc, ypc) in enumerate([('GCTA_PC1','GCTA_PC2'), ('GCTA_PC2','GCTA_PC3'), ('GCTA_PC3','GCTA_PC4')], start=1):
        fig.add_trace(dict(
            type='scatter',
            x=subset[xpc],
            y=subset[ypc],
            mode='markers',
            name=country,
            marker=dict(color=color, size=4),
            showlegend=False,
        ), row=row_idx, col=col_idx)

fig.update_xaxes(showline=True, linewidth=0.5, linecolor='black', mirror=True,
                 ticks='outside', tickwidth=0.5, tickcolor='black')
fig.update_yaxes(showline=True, linewidth=0.5, linecolor='black', mirror=True,
                 ticks='outside', tickwidth=0.5, tickcolor='black')

# Axis labels only on first row
fig.update_xaxes(title_text="PC1", row=1, col=1)
fig.update_yaxes(title_text="PC2", row=1, col=1)
fig.update_xaxes(title_text="PC2", row=1, col=2)
fig.update_yaxes(title_text="PC3", row=1, col=2)
fig.update_xaxes(title_text="PC3", row=1, col=3)
fig.update_yaxes(title_text="PC4", row=1, col=3)

# Square aspect per subplot
fig.update_layout(**{
    f'yaxis{i if i > 1 else ""}': dict(scaleanchor=f'x{i if i > 1 else ""}', scaleratio=1)
    for i in range(1, n * 3 + 1)
})

fig.update_layout(
    height=250 * n,
    width=900,
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=50, r=50, t=50, b=50),
    font=dict(family='Arial'),
)
# fig.write_image(f"{pca_dir}/pca_plot.svg")
fig.show()

In [ ]:
color_map = {
    'KAZ': '#636EFA',
    'AZE': '#EF553B',
    'GEO': '#00CC96',
    'ARM': '#AB63FA',
    'TJK': '#FFA15A',
}

countries = list(color_map.keys())
counts = pca['POP'].value_counts()[countries]

fig = go.Figure(go.Pie(
    labels=counts.index,
    values=counts.values,
    hole=0.4,
    marker=dict(
        colors=[color_map[c] for c in counts.index],
        line=dict(color='white', width=2),
    ),
    showlegend=False,
    textfont=dict(family='Arial', size=16),
))

for country in countries:
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        name=country,
        marker=dict(
            symbol='square',
            size=12,
            color=color_map[country],
            line=dict(color='black', width=1),
        ),
        showlegend=True,
    ))

fig.update_layout(
    height=450,
    width=500,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=16),
    margin=dict(l=50, r=50, t=50, b=50),
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
)

fig.show()

In [ ]:
ancestries = list(label_color_map.keys())
counts = gt['label'].value_counts().reindex(ancestries).fillna(0)
total = counts.sum()

mask = (counts / total) >= 0.10
texts = [f"{l}<br>{v/total:.1%}" if m else "" 
         for l, v, m in zip(counts.index, counts.values, mask)]

fig = go.Figure(go.Pie(
    labels=counts.index,
    values=counts.values,
    hole=0.4,
    marker=dict(
        colors=[label_color_map[c] for c in counts.index],
        line=dict(color='white', width=2),
    ),
    text=texts,
    textinfo="text",
    textposition="inside",
    insidetextorientation="horizontal",
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Count: %{value}<br>"
        "Fraction: %{percent}<extra></extra>"
    ),
    showlegend=False,
    textfont=dict(family='Arial', size=16),
))

for ancestry in ancestries:
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        name=ancestry,
        marker=dict(
            symbol='square',
            size=12,
            color=label_color_map[ancestry],
            line=dict(color='black', width=1),
        ),
        showlegend=True,
    ))

fig.update_layout(
    height=450,
    width=500,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=16),
    margin=dict(l=50, r=50, t=50, b=50),
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
)

fig.show()

In [ ]:
fig = make_subplots(
    rows=n, cols=3,
    subplot_titles=[f"{c}: PC{a} vs PC{b}" for c in countries for a, b in [(1,2),(2,3),(3,4)]],
    horizontal_spacing=0.1,
    vertical_spacing=0.08,
)

seen_labels = set()
for row_idx, country in enumerate(countries, start=1):
    subset = pca[pca['POP'] == country]

    for col_idx, (xpc, ypc) in enumerate([('GCTA_PC1','GCTA_PC2'), ('GCTA_PC2','GCTA_PC3'), ('GCTA_PC3','GCTA_PC4')], start=1):
        for label, group in subset.groupby('label'):
            show = (label not in seen_labels) and (col_idx == 1) and (row_idx == 1)
            fig.add_trace(dict(
                type='scatter',
                x=group[xpc],
                y=group[ypc],
                mode='markers',
                name=label,
                marker=dict(color=label_color_map.get(label, 'grey'), size=4),
                showlegend=show,
            ), row=row_idx, col=col_idx)
            if show:
                seen_labels.add(label)

fig.update_xaxes(showline=True, linewidth=0.5, linecolor='black', mirror=True,
                 ticks='outside', tickwidth=0.5, tickcolor='black')
fig.update_yaxes(showline=True, linewidth=0.5, linecolor='black', mirror=True,
                 ticks='outside', tickwidth=0.5, tickcolor='black')

fig.update_xaxes(title_text="PC1", row=1, col=1)
fig.update_yaxes(title_text="PC2", row=1, col=1)
fig.update_xaxes(title_text="PC2", row=1, col=2)
fig.update_yaxes(title_text="PC3", row=1, col=2)
fig.update_xaxes(title_text="PC3", row=1, col=3)
fig.update_yaxes(title_text="PC4", row=1, col=3)

fig.update_layout(**{
    f'yaxis{i if i > 1 else ""}': dict(scaleanchor=f'x{i if i > 1 else ""}', scaleratio=1)
    for i in range(1, n * 3 + 1)
})

# Hide legend from all real traces
for trace in fig.data:
    trace.showlegend = False

# Add dummy legend-only traces
for label, color in label_color_map.items():
    fig.add_trace(dict(
        type='scatter',
        x=[None], y=[None],
        mode='markers',
        name=label,
        marker=dict(
            symbol='square',
            size=12,
            color=color,
            line=dict(color='black', width=1),
        ),
        showlegend=True,
    ))
fig.update_layout(
    height=250 * n,
    width=900,
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=50, r=50, t=50, b=50),
    font=dict(family='Arial'),
)

fig.show()